In [9]:
import os
import re
import pandas as pd
import numpy as np
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from concurrent.futures import ThreadPoolExecutor
import random
import json
import time
from tqdm.notebook import tqdm  # Progress bar

In [10]:
# ----- 1. Optimalizált böngésző beállítások -----
def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")  # Háttérben futás
    options.add_argument("--disable-images")  # Képek letiltása
    options.add_argument("--blink-settings=imagesEnabled=false")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.page_load_strategy = 'eager'
    
    caps = webdriver.DesiredCapabilities.CHROME
    caps["pageLoadStrategy"] = "eager"  # Ne várja meg a teljes oldal betöltését
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.set_page_load_timeout(20)
    return driver

# ----- 2. Cache kezelés -----
CACHE_FILE = "scrape_cache.json"

def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE) as f:
            return json.load(f)
    return {"processed_urls": []}

def save_cache(url):
    cache = load_cache()
    if url not in cache["processed_urls"]:
        cache["processed_urls"].append(url)
        with open(CACHE_FILE, 'w') as f:
            json.dump(cache, f)

# ----- 3. Segédfüggvények -----
def extract_number(text):
    """Optimalizált számkinyerő"""
    if not text or pd.isna(text):
        return None
    text = str(text).replace("\xa0", "").replace(".", "").replace(" ", "")
    match = re.search(r'(\d+,?\d*)', text)
    return float(match.group(1).replace(",", ".")) if match else None

def smart_wait(driver, selector, timeout=3):
    """Intelligens várakozás elemre"""
    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, selector)))
        return True
    except:
        return False

# ----- 4. Adatkinyerés egy hirdetésből -----
def scrape_listing(url, driver):
    try:
        driver.get(url)
        
        # Alapvető adatok gyűjtése
        listing_data = {"url": url}
        
        # Intelligens várakozás a kulcselemekre
        if not smart_wait(driver, 'h1[data-id="h1"]'):
            return None
        
        # Cím
        try:
            listing_data["title"] = driver.find_element(By.CSS_SELECTOR, 'h1[data-id="h1"]').text.strip()
        except:
            listing_data["title"] = None
        
        # Ár
        try:
            price_el = driver.find_element(By.CSS_SELECTOR, 'div.fc-black-2.fs-32.fw-900')
            listing_data["price"] = extract_number(price_el.text)
        except:
            listing_data["price"] = None
        
        # Alapterület
        try:
            area_el = driver.find_element(By.CSS_SELECTOR, 'div[data-cy="advert-details-first-param"]')
            listing_data["area_m2"] = extract_number(area_el.text)
        except:
            listing_data["area_m2"] = None
        
        # Emelet
        try:
            floor_el = driver.find_element(By.CSS_SELECTOR, 'div.text-nowrap.fc-black-2.fs-20.fw-bold')
            listing_data["floor"] = floor_el.text.strip()
        except:
            listing_data["floor"] = None
        
        # Szobák
        try:
            rooms_el = driver.find_element(By.CSS_SELECTOR, 'div[data-cy="advert-details-second-param"]')
            listing_data["rooms"] = extract_number(rooms_el.text)
        except:
            listing_data["rooms"] = None
        
        # Helyszín
        try:
            loc_el = driver.find_element(By.CSS_SELECTOR, 'button[data-cy="advert-map-map-btn"] span.fs-16')
            listing_data["location"] = loc_el.text.strip()
        except:
            listing_data["location"] = None
        
        # További tulajdonságok
        properties = {}
        try:
            items = driver.find_elements(By.CSS_SELECTOR, 'div[data-cy="advert-details-param-list-item"]')
            for item in items:
                try:
                    key = item.find_element(By.CSS_SELECTOR, 'span').text.strip().replace(":", "")
                    val = item.find_element(By.CSS_SELECTOR, '.fw-bold').text.strip()
                    properties[key] = val
                except:
                    continue
        except:
            pass
        
        listing_data.update(properties)
        listing_data["scrape_date"] = datetime.now().strftime('%Y-%m-%d')
        
        return listing_data
    
    except Exception as e:
        print(f"Hiba a(z) {url} feldolgozásakor: {str(e)}")
        return None

# ----- 5. Párhuzamos feldolgozás -----
def process_batch(urls, existing_urls, max_workers=3):
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        drivers = [setup_driver() for _ in range(max_workers)]
        
        for i, url in enumerate(urls):
            if url in existing_urls or pd.isna(url):
                continue
                
            driver = drivers[i % max_workers]
            futures.append(executor.submit(scrape_listing, url, driver))
            time.sleep(random.uniform(0.5, 1.5))  # Véletlenszerű késleltetés
            
        for future in tqdm(futures, desc="Feldolgozás"):
            result = future.result()
            if result:
                results.append(result)
                save_cache(result["url"])
    
    for driver in drivers:
        driver.quit()
    
    return results

In [11]:
# ----- 6. Fő feldolgozó logika -----
def main():
    # Beolvasás és inicializálás
    links_df = pd.read_csv("zenga_links.csv")
    output_path = "zenga_listings_details2.csv"
    
    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        existing_urls = set(existing_df["url"].tolist())
    else:
        existing_df = pd.DataFrame()
        existing_urls = set()
    
    # Cache betöltése
    cache = load_cache()
    existing_urls.update(cache["processed_urls"])
    
    # Feldolgozandó URL-ek szűrése
    urls_to_process = [url for url in links_df["url"] if url not in existing_urls and not pd.isna(url)]
    print(f"Összesen {len(urls_to_process)} új hirdetés feldolgozásra vár.")
    
    # Batch feldolgozás
    batch_size = 50
    all_data = []
    
    for i in tqdm(range(0, len(urls_to_process), batch_size), desc="Teljes folyamat"):
        batch = urls_to_process[i:i+batch_size]
        batch_results = process_batch(batch, existing_urls)
        all_data.extend(batch_results)
        
        # Részeredmények mentése
        if all_data:
            new_df = pd.DataFrame(all_data)
            if not existing_df.empty:
                updated_df = pd.concat([existing_df, new_df], ignore_index=True)
            else:
                updated_df = new_df
            
            updated_df.to_csv(output_path, index=False)
            print(f"\n{len(all_data)} új hirdetés hozzáadva. Mentve: {output_path}")

# Futtatás
if __name__ == "__main__":
    main()

Összesen 2000 új hirdetés feldolgozásra vár.


Teljes folyamat:   0%|          | 0/40 [00:00<?, ?it/s]

Feldolgozás:   0%|          | 0/50 [00:00<?, ?it/s]


46 új hirdetés hozzáadva. Mentve: zenga_listings_details2.csv


Feldolgozás:   0%|          | 0/50 [00:00<?, ?it/s]

...


1724 új hirdetés hozzáadva. Mentve: zenga_listings_details2.csv


Feldolgozás:   0%|          | 0/50 [00:00<?, ?it/s]


1773 új hirdetés hozzáadva. Mentve: zenga_listings_details2.csv


Feldolgozás:   0%|          | 0/50 [00:00<?, ?it/s]


1823 új hirdetés hozzáadva. Mentve: zenga_listings_details2.csv
